# SI — TetraSpeck bead precision: the colour counting model, and the three excitation lasers

**Makes up:** two SI figures for SI Note *Calibrating Precision* (Fig. 2d):

1. **Counting model** (`BeadPrecision_CountingModel.svg`) — the colour precision of all three pixel
   types and all three lasers falls on one curve against $N_X/g_X$ (the axis of Fig. 2d): bead colour
   precision **a)** against the photoelectrons on each pixel type, $N_X$, and **b)** against $N_X/g_X$.
2. **By excitation laser** (`BeadPrecision_ByExcitation.svg`) — **a)** σ_xy against the
   density-normalised $N\,(s_\mathrm{ref}/s_a)^2$ (below), and **b–d)** each pixel type's σ_colour against
   $N_X/g_X$, binned per laser (488, 561, 638 nm), where the lasers' signal ranges overlap. If the lasers
   agree there, pooling them in Fig. 2d is justified (Reviewer 4, minor 7).

**Input:** `bead_precision.csv`, one row per bead, written by §3 of `../Figure_2d_TetraspeckPrecision.ipynb`
(run that first, with the same `OUT_ROOT` and `FAST_MODE`). The same cuts as Fig. 2d are applied
(`_beads.clean_loc`, `_beads.clean_colour`).

## The counting model

The fit returns, for each pixel type Y, the amplitude $A_Y$ that type would collect if it covered the
whole sensor. It is measured from the $N_Y = f_Y A_Y$ photoelectrons pixel type Y detects ($f_Y$: its share
of the sensor, $\tfrac14, \tfrac12, \tfrac14$ for the Bayer mosaic), so counting (Poisson) noise gives
$\mathrm{var}(A_Y) = A_Y/f_Y$. The colour is the share $a_X = A_X/\sum_Y A_Y$ (relative pixel QE).
Propagating (delta method, no background, independent pixel types):

$$\left(\frac{\sigma_{a_X}}{a_X}\right)^2 = \frac{g_X}{N_X},\qquad
g_X = (1-a_X)^2 + a_X f_X \sum_{Y\neq X}\frac{a_Y}{f_Y}.$$

With equal pixel fractions $g_X = 1-a_X$, the familiar binomial result. On the Bayer mosaic pixel 2
covers twice the area of pixels 1 and 3, and $g_2 > 1-a_2$ (1.7× for the 488 nm beads). Plotting
σ_colour against the effective count $N_X/g_X$ puts every pixel type on $1/\sqrt{N_X/g_X}$ for pure
counting noise. Background, read noise and the fit add a roughly constant factor on top, which the
$a/\sqrt{N}+c$ fit of Fig. 2d absorbs.

## Localisation precision and PSF width

For a Gaussian PSF of width $s$ on pixels of size $a$, least-squares fitting gives (Mortensen *et al.*,
*Nat. Methods* 2010)

$$\sigma_{xy}^2 = \frac{s_a^2}{N}\left(\frac{16}{9} + \frac{8\pi s_a^2 b}{N a^2}\right),\qquad s_a^2 = s^2 + \frac{a^2}{12},$$

with $b$ the background per pixel. With $D = N/s_a^2$ (photoelectrons per unit PSF area) and
$\beta = b/a^2$ this is $\sigma_{xy}^2 = (16/9 + 8\pi\beta/D)/D$: the precision depends on $N$ and
$s$ only through the photoelectron density. The beads excited at 638 nm emit further in the red and
have a wider PSF (median fitted $s$ ≈ 170 nm, against ≈ 135 and 150 nm at 488 and 561 nm), so the
same $N$ is spread over a larger area. Plotting against $N\,(s_\mathrm{ref}/s_a)^2$, the photoelectrons a
PSF of width $s_\mathrm{ref}$ would need for the same density, removes that; $s_\mathrm{ref}$ is the median
$s_a$ of all beads, so the axis stays close to the detected photoelectrons. Each bead uses its own
mean fitted width ($s = \sqrt{s_x s_y}$, which also includes the 100 nm bead size and any defocus).

Changed from the published Fig. 2d: σ_colour was plotted against the signal on each pixel type with
pixel 2 multiplied by $\sqrt2$; from 2026-11-09 to 2026-11-16 against $N_X/(1-a_X)$. The figures show only
the axes used in the paper; the printed numbers in §2 and §3 also compare $N_X/(1-a_X)$ and σ_xy against
the raw $N$.

---
## 0 — Imports and configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# _beads.py lives one folder up, beside Figure_2d_TetraspeckPrecision.ipynb.
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _beads as bd

from pyS3M import PlottingBase
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FAST_MODE = False   # as in Figure_2d_TetraspeckPrecision: which of its runs to read
OUT_ROOT  = Path("/media/u2473660/JSBLeeLabBackup/Data/20261026_Tetraspeck")   # Figure_2d's OUT_ROOT
BEAD_TABLE = OUT_ROOT / ("fast" if FAST_MODE else "full") / "bead_precision.csv"
FIG_DIR = OUT_ROOT / ("fast" if FAST_MODE else "full") / "SI_BeadPrecision"
FIG_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT = 1000
MIN_PER_BIN = 3 if FAST_MODE else 5
EDGES = np.geomspace(10, bd.MAX_PE, 16)

PIXEL_MARKERS = {"B": "o", "G": "s", "R": "^"}
PIXEL_NAMES = {"B": "pixel 1", "G": "pixel 2", "R": "pixel 3"}


def grid(ax):
    ax.grid(color="gray", lw=0.5, ls="--", alpha=0.5)
    ax.set_axisbelow(True)


def log_axes(ax, x=True, y=True):
    for on, axis in ((x, ax.xaxis), (y, ax.yaxis)):
        if on:
            axis.set_major_locator(mticker.LogLocator(base=10, numticks=12))
            axis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
            axis.set_minor_formatter(mticker.NullFormatter())


print(f"Bead table: {BEAD_TABLE}  ({'found' if BEAD_TABLE.exists() else 'NOT FOUND: run Figure_2d §3 first'})")

---
## 1 — Beads (same cuts as Fig. 2d)

In [ ]:
beads = pd.read_csv(BEAD_TABLE, dtype={"laser": str, "power": str})
if "s_a_nm" not in beads:
    raise KeyError(f"{BEAD_TABLE.name} has no PSF width (s_a_nm): rerun §3 of Figure_2d_TetraspeckPrecision")
loc = bd.clean_loc(beads)
S_REF_NM = float(np.median(loc["s_a_nm"]))   # reference PSF width: the median over the sigma_xy beads
loc = loc.assign(pe_norm=bd.density_normalised_pe(loc["pe"], loc["s_a_nm"], S_REF_NM),
                 mortensen=bd.mortensen_sigma(loc["pe"], loc["s_a_nm"], loc["bg_pe"]))
print(f"s_ref = {S_REF_NM:.1f} nm (median s_a of the sigma_xy beads)")
print(loc.groupby("laser").agg(beads=("pe", "size"), s_nm=("s_nm", "median"), s_a_nm=("s_a_nm", "median"),
                               bg_pe=("bg_pe", "median"))
      .assign(**{"sigma_xy / Mortensen": (loc["sigma_xy"] / loc["mortensen"]).groupby(loc["laser"]).median()})
      .round(2))
colour = bd.clean_colour(bd.colour_long(beads))
colour = colour.assign(n_1ma=colour["pe_px"] / (1 - colour["A_px"]))
print(f"{len(beads):,} beads; {len(loc):,} in the sigma_xy plot; {len(colour):,} bead-pixel pairs in the colour plot")

# expected shares of each laser's beads, from the dye spectra (as SI Table photons_to_pe)
SHARES = {}
for laser in bd.LASERS:
    _, q = bd.bead_effective_qe(laser)
    SHARES[laser] = {ch: q[ch] / sum(q.values()) for ch in bd.CHANNELS}
g_expected = pd.DataFrame({f"{l} nm": bd.colour_count_factor(SHARES[l]) for l in bd.LASERS}).T
one_minus = pd.DataFrame({f"{l} nm": {ch: 1 - SHARES[l][ch] for ch in bd.CHANNELS} for l in bd.LASERS}).T
print("Expected shares a_X:")
print(pd.DataFrame(SHARES).T.round(3))
print("\ng_X / (1 - a_X) at the expected shares:")
print((g_expected / one_minus).round(2))
print("\nMeasured median shares per laser:")
print(beads.groupby("laser")[[f"A_{c}" for c in bd.CHANNELS]].median().round(3))

---
## 2 — Figure: the counting model

Bead colour precision (median per bin of each laser × pixel type, 95 % bootstrap interval) **a)** against
$N_X$ and **b)** against $N_X/g_X$. Dashed: pure counting noise, $1/\sqrt{x}$. σ_colour is SD/mean of the relative pixel QE (a fraction).

Printed checks (not plotted): a Monte Carlo of the formula (for each laser's expected shares and
$N$ = 100–10,000 detected photoelectrons, 20,000 frames of Poisson photoelectrons on each pixel type, no
background, amplitudes $N_Y/f_Y$; simulated / predicted std of the shares), and the log-residual SD of
$\sigma_\mathrm{colour}\sqrt{x}$ over all bead-pixel pairs, which measures how well each axis collapses the nine
groups (also for $N_X/(1-a_X)$).

In [ ]:
mc = pd.concat([bd.simulate_colour_precision(SHARES[l], n, seed=k).assign(laser=l, n_pe=n)
                for l in bd.LASERS for k, n in enumerate(np.geomspace(100, 1e4, 5))], ignore_index=True)
print("Monte Carlo / formula, range over lasers, pixels and N:")
print(f"  exact g_X : {np.min(mc['simulated'] / mc['exact']):.3f} - {np.max(mc['simulated'] / mc['exact']):.3f}")
print(f"  1 - a_X   : {np.min(mc['simulated'] / mc['one_minus_a']):.3f} - {np.max(mc['simulated'] / mc['one_minus_a']):.3f}")

axes_x = [("pe_px", r"N$_{\mathrm{X}}$ (photoelectrons on pixel X)"),
          ("n_1ma", r"N$_{\mathrm{X}}$/(1$-$a$_{\mathrm{X}}$)"),
          ("n_eff", r"N$_{\mathrm{X}}$/g$_{\mathrm{X}}$")]
spread = {}
for key, _ in axes_x:
    c = colour[colour[key] > 0]
    r = np.log(c["prec"] * np.sqrt(c[key]))
    groups = np.exp(c.assign(r=r).groupby(["laser", "channel"])["r"].median())
    spread[key] = {"log-residual SD": r.std(), "group median min": groups.min(), "group median max": groups.max(),
                   "max / min": groups.max() / groups.min()}
print("\nsigma_colour x sqrt(x), over bead-pixel pairs (group = laser x pixel type):")
print(pd.DataFrame(spread).T.rename(index=dict((k, l) for k, l in axes_x)).round(3))

shown = [axes_x[0], axes_x[2]]   # N_X and N_X/g_X (Fig. 2d's axis)
fig, axs = plotter.two_column_plot(nrows=1, ncols=2, height=2.6)
for ax, (key, xlabel) in zip(axs, shown):
    for laser in bd.LASERS:
        for ch in bd.CHANNELS:
            e = colour[(colour["laser"] == laser) & (colour["channel"] == ch)]
            b = bd.binned_medians(e[key], e["prec"], EDGES, MIN_PER_BIN, N_BOOT).dropna()
            ax.errorbar(b["pe"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                        color=bd.LASER_COLOURS[laser], marker=PIXEL_MARKERS[ch], ms=3, lw=0.8, capsize=1.5)
    x = np.geomspace(EDGES[0], EDGES[-1], 50)
    ax.plot(x, 1 / np.sqrt(x), color="k", lw=0.8, ls="--")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(EDGES[0], EDGES[-1]); ax.set_ylim(0.005, 0.5)
    log_axes(ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$\mathrm{\sigma_{colour}}$/relative pixel QE")
    grid(ax)
axs[1].legend(handles=[Line2D([], [], color=bd.LASER_COLOURS[l], lw=1, label=f"{l} nm") for l in bd.LASERS]
              + [Line2D([], [], ls="", marker=PIXEL_MARKERS[c], color="k", ms=3, label=PIXEL_NAMES[c]) for c in bd.CHANNELS]
              + [Line2D([], [], color="k", lw=0.8, ls="--", label="counting noise")],
              fontsize=5, loc="lower left")
plt.savefig(FIG_DIR / "BeadPrecision_CountingModel.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()


---
## 3 — Figure: precision by excitation laser

Each laser's beads binned on $N\,(s_\mathrm{ref}/s_a)^2$ (**a**, σ_xy) or on that pixel type's $N_X/g_X$
(**b–d**, σ_colour), log-spaced bins; median and 95 % bootstrap interval for bins with ≥ 5 beads. The
lasers agree where their intervals overlap. Dashed line in **a**: Mortensen's precision for a PSF of
width $s_\mathrm{ref}$ at the median background, with no excess noise. The table lists the bins where at
least two lasers have ≥ 5 beads, and the largest ratio between the lasers' medians in each; it also
includes σ_xy against the raw $N$ (not plotted), for comparison.


In [ ]:
binned = {}
for laser in bd.LASERS:
    d = loc[loc["laser"] == laser]
    binned["sigma_xy", laser] = bd.binned_medians(d["pe"], d["sigma_xy"], EDGES, MIN_PER_BIN, N_BOOT)
    binned["sigma_xy_norm", laser] = bd.binned_medians(d["pe_norm"], d["sigma_xy"], EDGES, MIN_PER_BIN, N_BOOT)
    for ch in bd.CHANNELS:
        e = colour[(colour["laser"] == laser) & (colour["channel"] == ch)]
        binned[f"colour_{ch}", laser] = bd.binned_medians(e["n_eff"], e["prec"], EDGES, MIN_PER_BIN, N_BOOT)

panels = [("sigma_xy_norm", r"$\mathrm{\sigma_{xy}}$/nm", r"N$_{\mathrm{photoelectrons}}$ (s$_{\mathrm{ref}}$/s$_{\mathrm{a}}$)$^2$")] + [
    (f"colour_{ch}", rf"{PIXEL_NAMES[ch]} $\mathrm{{\sigma_{{colour}}}}$/relative pixel QE", r"N$_{\mathrm{X}}$/g$_{\mathrm{X}}$") for ch in bd.CHANNELS]
fig, axs = plotter.two_column_plot(nrows=1, ncols=4, height=2.2)
for ax, (key, ylabel, xlabel) in zip(axs, panels):
    for laser in bd.LASERS:
        b = binned[key, laser].dropna()
        ax.errorbar(b["pe"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                    color=bd.LASER_COLOURS[laser], marker="o", ms=3, lw=1, capsize=2, label=f"{laser} nm")
    ax.set_xscale("log")
    ax.set_xlim(100 if key.startswith("sigma_xy") else EDGES[0], EDGES[-1])
    log_axes(ax, y=False)
    ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    grid(ax)
x = np.geomspace(100, EDGES[-1], 100)
axs[0].plot(x, bd.mortensen_sigma(x, S_REF_NM, loc["bg_pe"].median()), color="k", lw=0.8, ls="--")
axs[0].set_ylim(0, 1.05 * max(np.nanmax(binned["sigma_xy_norm", l]["hi"]) for l in bd.LASERS))
axs[0].legend(fontsize=6)
plt.savefig(FIG_DIR / "BeadPrecision_ByExcitation.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

rows = []
for key in ["sigma_xy"] + [k for k, *_ in panels]:
    wide = pd.concat({l: binned[key, l].set_index("pe")[["n", "median", "lo", "hi"]] for l in bd.LASERS}, axis=1)
    med = wide.xs("median", axis=1, level=1)
    shared = med.notna().sum(axis=1) >= 2
    rows.append(wide[shared].assign(quantity=key, max_over_min=(med.max(axis=1) / med.min(axis=1))[shared]))
overlap = pd.concat(rows).reset_index().set_index(["quantity", "pe"])
overlap.to_csv(FIG_DIR / "overlap_binned_medians.csv")
print(overlap.groupby(level="quantity")["max_over_min"].agg(["count", "median", "max"]).round(2))
overlap.round(2)